[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/limorigu/uchicago-python-ai-workshop/blob/main/notebooks/03_news_dashboard_STARTER.ipynb)

# 03 · Case study 2 — A news screen for an investment target (Day 2)

**Situation.** Your team is evaluating **Marriott International** (NASDAQ: MAR) as a potential investment. Before the first call with management you want a quick, repeatable read of what the news says about the company — and, crucially, you want to know *when to trust that read.*

*(Why Marriott and not, say, Hilton? We tried Hilton first — and the feed drowned in headlines about Steve Hilton, who is running for governor of California next month. "Hilton" is also a person, an island, and a celebrity. That query ambiguity is a real lesson — you will meet it again below — but it makes a bad default target three weeks before an election.)*

**What you will build:** `fetch → clean → baseline sentiment → LLM classification → evaluate → iterate the prompt`.

| Part | When | Output |
|---|---|---|
| A. Fetch, clean, baseline | 10:30–11:10 | a DataFrame of headlines with a naive sentiment score, and a first look at where it is wrong |
| B. Gold labels + LLM run | 11:10–12:00 | your team's 20 hand labels, inter-annotator agreement, a first LLM classification |
| B (cont.) Evaluate + iterate | 1:10–1:40 | the accuracy table, the disagreement list, and a **required** `PROMPT_V3` with a logged score |

**Teamwork rule (teams of 5, finance + tech).** Tech students drive the code; finance students drive every *judgement* — what counts as relevant, what an investor calls negative, which disagreements matter. Rotate the keyboard at Part B. The sprint rubric this afternoon scores exactly this split.

**About `newslib`.** The pipeline's plumbing lives in `app/newslib.py` — roughly 300 lines of Python **written for this workshop**. It is *not* an installed package like `pandas` or `numpy`: it is a plain `.py` file sitting in this repository, which is why the setup cell adds the `app/` folder to Python's path before `import newslib` works. You are encouraged to open it and read it. It follows exactly the library-plus-tests pattern the finance cohort pushed through GitHub Actions yesterday with `case1/whitespace.py` — same idea, different case. Today nobody builds plumbing; you build judgement on top of it.

**No API key?** Every cell runs in *mock mode* (a rule-based stand-in). Your team can also run the prompt in a chat UI on 20 headlines by hand — see Part B.

**If the live feed is blocked** on the classroom network, the notebook automatically switches to a *synthetic* feed about a *fictional* hotel company, "Harborview Hotels", so the pipeline can still be exercised. A banner will tell you.

In [ ]:
# --- Setup: get the workshop files into this Colab session ------------------------------
# Downloads the workshop repository (data + helper code). Takes ~10 seconds the first time.
import os, sys
REPO_URL = "https://github.com/limorigu/uchicago-python-ai-workshop"
if not any(os.path.exists(d) for d in ["workshop/data", "data", "../data"]):
    !GIT_TERMINAL_PROMPT=0 git clone -q --depth 1 $REPO_URL workshop
DATA = next((d for d in ["workshop/data", "data", "../data"] if os.path.exists(d)), None)
if DATA is None:
    raise RuntimeError("Could not download the workshop files. Check your internet connection and "
                       "that this link opens in your browser: " + REPO_URL + " . Still stuck? Email the instructor.")
sys.path.insert(0, "workshop/app" if os.path.exists("workshop/app") else "../app")
print("Data folder:", DATA, "| files:", sorted(os.listdir(DATA)))

## Part A · Step 1 — Fetch
Google News publishes an **RSS feed** for any search — a machine-readable version of the search results that anyone can download, no API key needed. RSS is written in **XML**, a text format (like HTML's stricter cousin) where each headline sits inside tags: `<item><title>…</title><source>…</source><pubDate>…</pubDate></item>`.

`newslib.fetch_google_news` does two things: builds the search URL, then walks the XML and pulls out one row per `<item>`. Remember: `newslib` is our workshop file, not a package — if you want to see the ~25 lines that do this, open `app/newslib.py` in the repo (or run `nl.fetch_google_news??` in a cell to print the source right here).

In [ ]:
import pandas as pd
import newslib as nl

COMPANY = "Marriott"
DESCRIPTION = "global hotel franchisor, NASDAQ: MAR"
EXTRA = 'when:30d -library'      # Google search syntax: time window + exclusions (the Marriott Library is a thing)

df = nl.fetch_with_fallback(COMPANY, f"{DATA}/case2/news_sample_cache.csv", extra_terms=EXTRA)
if df.get("is_synthetic", pd.Series([False])).any():
    COMPANY, DESCRIPTION = "Harborview Hotels", "fictional global hotel franchisor (synthetic classroom feed)"
print(len(df), "headlines about", COMPANY)
df.head()

**Look at the raw titles before doing anything else.** What is *not* about the company? What is duplicated? What would fool a keyword approach?

In [ ]:
pd.set_option("display.max_colwidth", 120)
df[["published", "source", "title"]].head(25)

In [ ]:
# OPTIONAL (live feed only): see why we abandoned Hilton as the target.
# df_hilton = nl.fetch_google_news("Hilton", extra_terms="when:7d")
# df_hilton.title.head(20)   # how many are about the hotel company vs the candidate for governor?

## Step 2 — Clean

Your TODO below calls `nl.clean_headlines(df)`. Before (or after) you run it, **figure out what the function actually does** — pick whichever of these you'd reach for on the job:

- **Ask the built-in AI.** Colab has Gemini built in (the ✨ *Ask Gemini* button, or the chat icon in the left sidebar): paste the function name and ask "what does `clean_headlines` in the open `newslib.py` do, step by step?" Or paste the function into whatever assistant you have open in another window (Claude, ChatGPT).
- **Read the source yourself.** Run `nl.clean_headlines??` in a cell (two question marks prints the full source), or open `app/newslib.py` via Colab's file browser (📁 icon in the left sidebar → `workshop/app/newslib.py`, double-click to view).

Then verify your understanding against the data: the row counts before/after *are* the answer key.

In [ ]:
# TODO: call nl.clean_headlines(df) and compare row counts before/after. What did it remove?

## Step 3 — Baseline: a naive lexicon
`newslib.lexicon_sentiment` counts positive and negative words from a small finance word list (real work would use the Loughran–McDonald dictionary). It is deliberately simple. **Your job is to find where it breaks.**

Expect a lot of neutrals on the live feed: real headlines hedge ("weighs", "expected to", "in talks") and the word list is small. That is itself a finding — a baseline that calls almost everything neutral can still score *high accuracy* if your gold labels are mostly neutral too, which is exactly why you will look at per-class precision/recall this afternoon, not just accuracy.

In [ ]:
df = nl.add_lexicon_sentiment(df)
df["lex_label"].value_counts()

In [ ]:
# TODO: show the 10 most positive and 10 most negative headlines by lex_score. Do you agree with them?
# TODO: find at least 3 headlines where the lexicon is clearly wrong and write down WHY (negation? sarcasm? finance jargon?
#       a word that is positive in general English but negative for an investor?)

## Part B · Step 4 — Hand labels (the gold set) — 15 min, team
Your team labels **15 headlines**: ten that *every* team labels (the common block) plus five that only your team covers. The cell below slices them out of the cleaned frame by your team number.

Each of the five of you labels all 15 **independently, without talking** (relevant / sentiment / category). Then compare: the share where you agree is your **inter-annotator agreement** — no model can be measured more precisely than that. Argue out a final team label for each headline.

**Class pool:** one person per team enters your 15 agreed rows into the shared class sheet — **the link is on the Case 2 slide being projected right now.** The design: 10 teams × the same common block = ten independent labels for each of those ten headlines (that's the class-wide agreement number for the debrief), and 10 teams × 5 unique = 50 more headlines of coverage. By lunch the room owns a ~60-headline gold set with a measured ceiling.

Categories: `earnings, m&a_or_capital, management, demand_or_macro, legal_or_regulatory, product_or_brand, esg_or_labor, other`.
Sentiment is **from an investor's perspective**: a hotel opening is routine → neutral; a CEO departure → usually negative.

In [ ]:
TEAM = 1   # <-- your team number (1-10, from the teams slide)

# Deterministic assignment: sort alphabetically so every team slices the same deck.
pool = df.sort_values("title").reset_index(drop=True)
common = pool.iloc[:10]                                  # every team labels these ten
mine = pool.iloc[10 + (TEAM - 1) * 5 : 10 + TEAM * 5]    # five more, unique to your team
to_label = pd.concat([common, mine])
to_label_out = to_label.reindex(columns=["id", "title"]).assign(relevant="", sentiment="", category="")
to_label_out.to_csv("team_labels_blank.csv", index=False)
print("Wrote team_labels_blank.csv —", len(to_label), "headlines. Each of you labels all of them, separately.")
to_label[["title"]]

In [ ]:
# After the team has argued out one agreed label per headline, save the result as our_labels.csv
# and upload it to this Colab session (folder icon on the left -> upload). Then re-run this cell.
import os
gold_path = "our_labels.csv"
if os.path.exists(gold_path):
    gold = pd.read_csv(gold_path)
    key = "id" if "id" in df.columns and "id" in gold.columns else "title"
    m = df.merge(gold[[key, "relevant", "sentiment", "category"]], on=key, how="inner", suffixes=("", "_gold"))
    print(len(m), "labelled headlines")
else:
    print("No labels yet — label your 15 headlines, save as our_labels.csv, upload, re-run.")

In [ ]:
# If two group members labelled separately, measure agreement:
# a = pd.read_csv("labels_alice.csv"); b = pd.read_csv("labels_bob.csv")
# print("human-human agreement on sentiment:", nl.agreement_rate(a.sentiment, b.sentiment))

## Step 5 — LLM classification
`newslib.PROMPT_V1` is what most people write first. `PROMPT_V2` applies this morning's **context-engineering** lesson — every line in it earned its place in the window: a role and an investor's perspective (so the model knows *whose* judgement to apply), an explicit relevance check (the exit ramp for headlines about the wrong Marriott), a closed category list (no invented labels), a JSON schema (so code can read the answer), and a rationale field (so *you* can read the model's reasoning when it disagrees with you).

**Track A (you brought your own API key — via your university's AI access):** set `ANTHROPIC_API_KEY` in Colab's *Secrets* panel (🔑 icon) — never paste it in a cell — and create the client below. It's your key and your spend — worth batching calls and using a cheaper model while you iterate.
**Track B (no key):** leave `client = None` (mock mode), *and* have one group member paste `PROMPT_V2` + 20 headlines into a chat UI, record the JSON answers into your labels file as `llm_sentiment`, and compare by hand.

In [ ]:
client = None
try:
    from google.colab import userdata          # Colab secrets
    # .strip(): a stray newline from the paste would make every API call fail
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY").strip()
except Exception:
    pass
if os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = os.environ["ANTHROPIC_API_KEY"].strip()
    !pip -q install anthropic
    import anthropic
    client = anthropic.Anthropic()
    print("Real model mode")
else:
    print("Mock mode — deterministic rule-based stand-in (prompt text is ignored in this mode)")
print(nl.PROMPT_V2.format(company=COMPANY, description=DESCRIPTION, headline="<headline>", source="<source>", categories=nl.CATEGORIES))

In [ ]:
# TODO: classify the frame with nl.classify_frame(df, COMPANY, DESCRIPTION, client=client) and look at the new llm_* columns

## Step 6 — Evaluate: lexicon vs LLM vs humans
Accuracy alone hides the interesting part. Look at the confusion matrix and the per-class report: *which* mistakes does each method make?

In [ ]:
# TODO (if labels exist): merge llm columns onto m and compute
#   nl.accuracy(m.lex_label[rel], m.sentiment[rel]) and nl.accuracy(m.llm_sentiment[rel], m.sentiment[rel]) where rel = m.relevant
#   nl.accuracy(m.llm_relevant.astype(bool), m.relevant)
#   nl.confusion(...), nl.per_class_report(...)

In [ ]:
# The most useful table in the whole exercise: where the model disagrees with the humans.
if "m" in globals() and "llm_sentiment" in m.columns:
    rel = m.relevant.astype(bool)
    dis = m[rel & (m.llm_sentiment != m.sentiment)][["title", "sentiment", "llm_sentiment", "llm_rationale"]]
    print(len(dis), "disagreements"); display(dis)
else:
    print("Finish the classification and evaluation TODOs above first.")

## Step 7 — Iterate the prompt (REQUIRED — every team ships a V3)
1. **State the change before you run it.** Pick *one* thing V2 gets wrong (read your disagreement list), write one sentence: "V3 adds ___ because V2 ___".
2. Copy `PROMPT_V2` into `PROMPT_V3`, make exactly that one change (e.g. two labelled examples of the hard case, or a rule for 'cost cuts').
3. Re-run and re-score. **Log both numbers.** Keep V3 even if it is *worse* — say why. A logged regression teaches more than an unlogged win.

**Track A:** re-run `classify_frame(..., prompt_template=PROMPT_V3)`.
**Track B (mock mode ignores prompt text):** run V2 and V3 by hand in a chat UI on the same 20 headlines and score both against your labels — the discipline is identical.

In [ ]:
# V3_CHANGE = "adds two labelled examples of cost-cut headlines, because V2 calls them negative and our gold says neutral"
# PROMPT_V3 = nl.PROMPT_V2 + "\n\nExamples:\n- 'Company cuts 200 corporate roles in efficiency push' -> sentiment neutral (cost discipline)\n"
# df_v3 = nl.classify_frame(df, COMPANY, DESCRIPTION, client=client, prompt_template=PROMPT_V3)
# TODO: re-compute accuracy on df_v3, then log:  print(f"V2: {acc_v2:.0%}  V3: {acc_v3:.0%}  change: {V3_CHANGE}")

## ⚡ Fast lanes (tech students: pick one while the team writes the debrief notes)
1. **Batching.** `classify_frame` calls the model once per headline. Batch 10 headlines per call (one JSON array out) and compare wall-clock time and accuracy.
2. **Model swap.** Re-run with a smaller/cheaper model (change the `model=` argument in `newslib.classify_headline`). Build a 2×2: accuracy vs cost.
3. **Cost tracker.** Count input/output tokens per call (the API response reports usage) and print cost-per-100-headlines.
4. **Dashboard.** Run the Streamlit app (cells below) — and deploy it on Streamlit Community Cloud if you finish early.

## Step 8 (fast lane) — Run the dashboard
`app/news_dashboard.py` is a Streamlit app that calls the same `newslib` functions. **Locally:** `streamlit run app/news_dashboard.py`. **In Colab:** run the two cells below — the first starts the server in the background, the second gives you a URL proxied through Colab (no third-party tunnel needed).

In [ ]:
APP_DIR = "workshop/app" if os.path.exists("workshop/app") else "../app"
!pip -q install streamlit
# system_raw truly detaches the server. (A plain `!command &` hangs the cell:
# the server keeps the cell's terminal open and Colab waits for it to close.)
# CORS/XSRF off: Colab's proxy serves the page from a different origin, and
# with the protections on, Streamlit rejects the websocket — the page loads
# as empty grey blocks and never renders.
get_ipython().system_raw(
    f"cd {APP_DIR} && streamlit run news_dashboard.py "
    "--server.port 8501 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false "
    "> /tmp/streamlit.log 2>&1 &")
import time; time.sleep(5)
print(open("/tmp/streamlit.log").read()[-500:] if os.path.exists("/tmp/streamlit.log") else "server log not found — run this cell in Colab")

In [ ]:
try:
    from google.colab.output import eval_js
    print("Open your dashboard:", eval_js("google.colab.kernel.proxyPort(8501)"))
except Exception:
    print("Not in Colab — open http://localhost:8501")

## A note on version control (done on Day 1)
The finance cohort lived this loop yesterday with Case 1: they pushed `case1/whitespace.py` plus its tests to GitHub, watched Actions go green, broke a function on purpose, watched the red ✗ catch it, and fixed it. `app/newslib.py` follows exactly the same library-plus-tests pattern (its tests live in `app/tests/`), so the same safety net is one push away for today's prompt work: one repo per team, `app/` + `.github/` at the root, and every `PROMPT_V3` committed with its score in the message. Tech students — if a teammate's test idea captures a failure you found today, add it: `assert lexicon_sentiment("record impairment")["lex_label"] != "positive"` is the house style.

## Case 2 debrief prep (before 1:40)
Have ready: your accuracy table (lexicon vs LLM vs gold, with n), your agreement rate, one disagreement you found interesting, and your V2→V3 numbers with the one-sentence change rationale.